In [ ]:
%pip install -r requirements.txt

In [ ]:
# Set up
import os
import sys
import subprocess
import itertools

sys.path.insert(0, os.path.abspath("sandbagging"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


In [ ]:
import torch
import config as config
import data as data
import prompts as prompts
import directions as directions
import evaluate as evaluate
import helpers as helpers
from hooks import (add_hooks, get_activation_addition_input_pre_hook,
                       get_all_direction_ablation_hooks,
                       get_mean_matching_pre_hook, get_span_zero_hook)
from model import load_model_tokenizer, generate

data.build()
problems, splits = data.load()

model, tok = load_model_tokenizer()


In [ ]:
# number of problems
N = 250
# Take N problems from test split
sample = data.subset(problems, splits, "test")[:N]

for condition in ["locked", "unlocked"]:
    # build N prompts, run model and grade answers
    acc, stderr, generations = helpers.score(model, tok, sample, condition)
    print(f"{condition:20s} {acc:.3f} +- {stderr:.3f}")
    print(generations[0])

In [ ]:
CONDITIONS = ["unlocked", "decoy", "locked"]

# problems reserved for computing direction
train = data.subset(problems, splits, "train")
# build three prompt sets from the same N problems
prompt_sets = {condition: prompts.make_prompts(train, tok, condition)
               for condition in CONDITIONS}

# for each prompt set, one fwd pass per prompt, get last token activation at each layer and avg across N problems
# shape [1, 30, 4096]
means = {condition: directions.get_mean_activations(model, tok, prompt_set)
         for condition, prompt_set in prompt_sets.items()}

# directions
d_none = means["unlocked"] - means["locked"]
d_decoy = means["unlocked"] - means["decoy"]

# grows w/ depth as activation magnitudes grows, one length per layer
norms = d_none[0].norm(dim=-1)
# check cosine similarity between directions
cos = torch.nn.functional.cosine_similarity(d_decoy, d_none, dim=-1)[0]

print(f"{'layer':>5} {'||r||':>9} {'decoy~none':>11}")
for layer in range(d_none.shape[1]):
    print(f"{layer:>5} {float(norms[layer]):>9.2f} {float(cos[layer]):>11.3f}")

In [ ]:
# Take problems from validation split (64)
val = data.subset(problems, splits, "val")[:config.N_VAL_SET]

# calculate accuracy measured against validation problems
locked_acc, _, _ = helpers.score(model, tok, val, "locked")
unlocked_acc, _, _ = helpers.score(model, tok, val, "unlocked")
print(f"locked_acc: {locked_acc}\tunlocked_acc: {unlocked_acc}")

# skip later 0 since direction is 0. Stop at 0.8*n_layers since late-layer directions resemble unembedding rows (Arditi et al)
# so there may be token suppression rather than actually acting on the lock
for layer in range(1, int(0.8 * config.N_LAYERS)):
    # pull layer's direction out of d_none
    vector = d_none[0, layer].to(model.device, model.dtype)
    # build hook with coefficient 1.0
    pre = [(model.model.layers[layer],
            get_activation_addition_input_pre_hook(vector, 1.0))]
    # measure
    acc, stderr, _ = helpers.score(model, tok, val, "locked", pre, [])
    # capability unlocking measurement metric
    p = (acc - locked_acc) / (unlocked_acc - locked_acc)
    print(f"layer {layer:2d}: {acc:.3f} +- {stderr:.3f}  p={p:+.2f}  "
          f"||r||={float(vector.norm()):.2f}")

for a, b in itertools.combinations([6, 7, 8, 9, 10], 2):
    cos = torch.nn.functional.cosine_similarity(
        d_none[0, a], d_none[0, b], dim=0)
    print(f"layer {a} ~ layer {b}: {float(cos):+.3f}")

In [ ]:
val_sel = data.subset(problems, splits, "val")[64:264]
locked_sel, _, _ = helpers.score(model, tok, val_sel, "locked")
unlocked_sel, _, _ = helpers.score(model, tok, val_sel, "unlocked")
print(f"Locked: {locked_sel:.3f}  Unlocked: {unlocked_sel:.3f}\n")

for layer in [6, 7, 8, 9, 10, 11, 12]:
    for coeff in [1.0, 1.5]:
        vector = d_none[0, layer].to(model.device, model.dtype)
        pre = [(model.model.layers[layer],
                get_activation_addition_input_pre_hook(vector, coeff))]
        acc, stderr, _ = helpers.score(model, tok, val_sel, "locked", pre, [])
        rho = (acc - locked_sel) / (unlocked_sel - locked_sel)
        print(
            f"layer {layer:2d} coeff {coeff}: {acc:.3f} +- {stderr:.3f}  p={rho:+.2f}")

In [ ]:
val_200 = data.subset(problems, splits, "val")[:200]
locked_acc_200, _, _ = helpers.score(model, tok, val_200, "locked")
unlocked_acc_200, _, _ = helpers.score(model, tok, val_200, "unlocked")
print(f"floor {locked_acc_200:.3f}  ceiling {unlocked_acc_200:.3f}")

In [ ]:
def p(acc):
    return (acc - locked_acc_200) / (unlocked_acc_200 - locked_acc_200)

In [ ]:
# the single source for every clamp target below
unit6 = torch.nn.functional.normalize(
    d_none[0, 6].float(), dim=0).to(model.device, model.dtype)
unit9 = torch.nn.functional.normalize(
    d_none[0, 9].float(), dim=0).to(model.device, model.dtype)
unit6_cpu = unit6.float().cpu()
unit9_cpu = unit9.float().cpu()

# s6 is prompt's position at the lock direction at layer 6 (vector d6 from exp2)
s6_locked = float(means["locked"][0, 6].float().cpu() @ unit6_cpu)
s6_unlocked = float(means["unlocked"][0, 6].float().cpu() @ unit6_cpu)
s9_locked = float(means["locked"][0, 9].float().cpu() @ unit9_cpu)
s9_unlocked = float(means["unlocked"][0, 9].float().cpu() @ unit9_cpu)
print(f"s6: locked {s6_locked:+.2f}  unlocked {s6_unlocked:+.2f}")
print(f"s9: locked {s9_locked:+.2f}  unlocked {s9_unlocked:+.2f}\n")

mean_matched_pre = [(model.model.layers[6],
                     get_mean_matching_pre_hook(unit6, s6_unlocked))]

acc, stderr, _ = helpers.score(
    model, tok, val_200, "locked", mean_matched_pre, [])
print(f"mean-matched, locked    {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

acc, stderr, _ = helpers.score(
    model, tok, val_200, "unlocked", mean_matched_pre, [])
print(f"mean-matched, unlocked  {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

In [ ]:
# random controls - adds random vector
vector_l6 = d_none[0, 6].to(model.device, model.dtype)

# 3 random seeds
for seed in range(3):
    generator = torch.Generator().manual_seed(seed)
    # 4096 random numbers normally distributed
    random_vector = torch.randn(
        vector_l6.shape, generator=generator).to(vector_l6)
    # norm matching - same length as the real direction
    random_vector = random_vector / random_vector.norm() * vector_l6.norm()
    # addition experiment
    pre = [(model.model.layers[6],
            get_activation_addition_input_pre_hook(random_vector, 1.5))]
    acc, stderr, _ = helpers.score(model, tok, val_200, "locked", pre, [])
    print(f"random seed {seed}: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

In [ ]:
vector_l9 = d_none[0, 9].to(model.device, model.dtype)
for seed in range(3):
    g = torch.Generator().manual_seed(seed)
    rv = torch.randn(vector_l9.shape, generator=g).to(vector_l9)
    # norm-matched to 29.6
    rv = rv / rv.norm() * vector_l9.norm()
    pre = [(model.model.layers[9],
            get_activation_addition_input_pre_hook(rv, 1.0))]
    acc, stderr, _ = helpers.score(model, tok, val_200, "locked", pre, [])
    print(f"random L9 seed {seed}: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

In [ ]:
for seed in range(3):
    g = torch.Generator().manual_seed(seed)
    u = torch.randn(d_none[0, 6].shape, generator=g).to(
        d_none[0, 6].to(model.device, model.dtype))
    u = u / u.norm()
    s_u = float(means["locked"][0, 6].float().cpu() @ u.float().cpu())
    pre = [(model.model.layers[6],
            get_mean_matching_pre_hook(u, s_u + (s6_unlocked - s6_locked)))]
    acc, stderr, _ = helpers.score(model, tok, val_200, "locked", pre, [])
    print(f"random clamp {seed}: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

In [ ]:
for coeff in [-1.0, -1.5]:
    pre = [(model.model.layers[6],
            get_activation_addition_input_pre_hook(vector_l6, coeff))]
    acc, stderr, generations = helpers.score(model, tok, val_200, "unlocked",
                                             pre, [])
    print(f"relock {coeff:>5}: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}"
          f"lands at {s6_unlocked + coeff * float(vector_l6.norm()):+.2f}")

In [ ]:
vector_l6 = d_none[0, 6].to(model.device, model.dtype)
prompts_unlocked_val = prompts.make_prompts(val_200, tok, "unlocked")
for coeff in [-1.0, -1.5]:
    pre = [(model.model.layers[6],
            get_activation_addition_input_pre_hook(vector_l6, coeff))]
    s9 = directions.get_coordinate(model, tok, prompts_unlocked_val, 9,
                                   unit9_cpu, pre)

In [ ]:
vector_l9 = d_none[0, 9].to(model.device, model.dtype)
for coeff in [-1.0, -1.5]:
    pre = [(model.model.layers[9],
            get_activation_addition_input_pre_hook(vector_l9, coeff))]
    acc, stderr, generations = helpers.score(model, tok, val_200, "unlocked",
                                             pre, [])

In [ ]:
val_200 = data.subset(problems, splits, "val")[:200]
locked_acc_200, _, _ = helpers.score(model, tok, val_200, "locked")
unlocked_acc_200, _, _ = helpers.score(model, tok, val_200, "unlocked")

# s6* = -6.13 is exactly where addition alpha=-1.5 lands the mean
targets_l6_unlocked = [-15.0, -10.0, -6.13, s6_locked, 0.0, 4.0, 6.0, 12.0]

for target in targets_l6_unlocked:
    pre = [(model.model.layers[6],
            get_mean_matching_pre_hook(unit6, float(target)))]
    acc, stderr, _ = helpers.score(model, tok, val_200, "unlocked",
                                   pre, [])
    print(f"s6 := {target:>7.2f}: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

prompts_unlocked_val = prompts.make_prompts(val_200, tok, "unlocked")
for target in targets_l6_unlocked:
    pre = [(model.model.layers[6],
            get_mean_matching_pre_hook(unit6, float(target)))]
    s9 = directions.get_coordinate(
        model, tok, prompts_unlocked_val, 9, unit9_cpu, pre)
    print(f"s6 := {target:>7.2f}, s9 = {s9:+6.2f}")

In [ ]:
LAYER = 6
# Get layer 6 direction
vector = d_none[0, LAYER].to(model.device, model.dtype)

# build all 90 hooks, 30 on layer inputs, 30 on attention outputs, 30 on MLP outputs
abl_pre, abl_fwd = get_all_direction_ablation_hooks(model, vector)
for condition in ["locked", "unlocked"]:
    acc, stderr, _ = helpers.score(
        model, tok, val_200, condition, abl_pre, abl_fwd)
    print(
        f"ablation, {condition:9s} {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

vector_l9 = d_none[0, 9].to(model.device, model.dtype)
ablation_pre, ablation_fwd = get_all_direction_ablation_hooks(model, vector_l9)

for condition in ["locked", "unlocked"]:
    acc, stderr, _ = helpers.score(model, tok, val_200, condition,
                                   ablation_pre, ablation_fwd)
    print(f"L9 ablation, {condition:9s} {acc:.3f} +- {stderr:.3f}  "
          f"p={p(acc):+.2f}")

In [ ]:
for seed in range(2):
    g = torch.Generator().manual_seed(100 + seed)
    u = torch.randn(d_none[0, 6].shape, generator=g).to(
        d_none[0, 6].to(model.device, model.dtype))
    rpre, rfwd = get_all_direction_ablation_hooks(model, u)
    for condition in ["locked", "unlocked"]:
        acc, stderr, _ = helpers.score(
            model, tok, val_200, condition, rpre, rfwd)
        print(f"random ablation s{seed}, {condition:9s} "
              f"{acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

In [ ]:
targets = [-5.0, s6_locked, 0.0, 2.0, 4.0, 5.0, 6.0, s6_unlocked, 10.0, 12.0]

for target in targets:
    pre = [(model.model.layers[6],
            get_mean_matching_pre_hook(unit6, float(target)))]
    acc, stderr, _ = helpers.score(model, tok, val_200, "locked", pre, [])
    print(f"target {target:>6.2f}: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

prompts_locked_val = prompts.make_prompts(val_200, tok, "locked")
for target in targets:
    pre = [(model.model.layers[6],
            get_mean_matching_pre_hook(unit6, float(target)))]
    s9 = directions.get_coordinate(
        model, tok, prompts_locked_val, 9, unit9_cpu, pre)
    print(f"s6 := {target:>7.2f}  ->  s9 = {s9:+6.2f}")

In [ ]:
targets_l9 = [s9_locked, -20.0, -15.0, -10.0, -7.0, -5.0, -3.0, -1.0, 0.0,
              2.0, s9_unlocked, 10.0]

for target in targets_l9:
    pre = [(model.model.layers[9],
            get_mean_matching_pre_hook(unit9, float(target)))]
    acc, stderr, _ = helpers.score(model, tok, val_200, "locked", pre, [])
    print(f"s9 := {target:>7.2f}: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

In [ ]:
g = torch.Generator().manual_seed(0)
rv = torch.randn(vector_l6.shape, generator=g).to(vector_l6)
rv = rv / rv.norm() * vector_l6.norm()
pre = [(model.model.layers[6],
        get_activation_addition_input_pre_hook(rv, -1.5))]
acc, stderr, gens = helpers.score(model, tok, val_200, "unlocked", pre, [])
print(f"random relock a=-1.5: {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")
print(gens[0][:400])

In [ ]:
val_200 = data.subset(problems, splits, "val")[:200]
SUBSET15 = val_200
floor_pins, ceiling_pins = locked_acc_200, unlocked_acc_200
print(f"floor {floor_pins:.3f}  ceiling {ceiling_pins:.3f}\n")

s6_pins = {"s6:=none": None, "s6:=lock": s6_locked, "s6:=unlock": s6_unlocked}
s9_pins = {"s9:=none": None, "s9:=lock": s9_locked, "s9:=unlock": s9_unlocked}

for condition in ["locked", "unlocked"]:
    for name6, target6 in s6_pins.items():
        for name9, target9 in s9_pins.items():
            if target6 is None and target9 is None:
                continue
            pre = []
            if target6 is not None:
                pre.append((model.model.layers[6],
                            get_mean_matching_pre_hook(unit6, target6)))
            if target9 is not None:
                pre.append((model.model.layers[9],
                            get_mean_matching_pre_hook(unit9, target9)))
            key = f"{condition} | {name6} | {name9}"
            acc, stderr, _ = helpers.score(model, tok, SUBSET15, condition,
                                           pre, [])
            print(f"{key:40s} {acc:.3f} +- {stderr:.3f}  p={p(acc):+.2f}")

In [ ]:
@torch.no_grad()
def get_writes(model, tok, prompt_list, layers, direction, batch_size=None,
               check=False):
    """Every writer's contribution onto the direction at the last prompt token (at all layers).
    """
    batch_size = batch_size or config.BATCH_SIZE
    n_heads = model.config.num_attention_heads
    d_model = model.config.hidden_size
    d_head = d_model // n_heads
    direction_f = direction.float()

    heads = {layer: [] for layer in layers}
    mlp = {layer: [] for layer in layers}

    def make_attn_hook(layer):
        W_O = model.model.layers[layer].self_attn.o_proj.weight.detach(
        ).float()
        W_O_heads = W_O.reshape(d_model, n_heads, d_head)

        def hook_fn(module, args):
            z = args[0][:, -1, :].float()
            z_heads = z.reshape(z.shape[0], n_heads, d_head)
            per_head = torch.einsum("mhd,bhd->bhm", W_O_heads, z_heads)
            proj = per_head @ direction_f
            if check:
                full = (z @ W_O.T) @ direction_f
            heads[layer].append(proj.cpu())
        return hook_fn

    def make_mlp_hook(layer):
        # the mlp's output is what it adds to the residual stream
        def hook_fn(module, args, output):
            mlp[layer].append(
                (output[:, -1, :].float() @ direction_f).detach().cpu())
        return hook_fn

    pre = [(model.model.layers[layer].self_attn.o_proj, make_attn_hook(layer))
           for layer in layers]
    fwd = [(model.model.layers[layer].mlp, make_mlp_hook(layer))
           for layer in layers]

    for start in range(0, len(prompt_list), batch_size):
        batch = tok(prompt_list[start:start + batch_size],
                    return_tensors="pt", padding=True).to(model.device)
        with add_hooks(pre, fwd):
            model(input_ids=batch.input_ids,
                  attention_mask=batch.attention_mask)
    return {layer: {"heads": torch.cat(heads[layer]),
                    "mlp": torch.cat(mlp[layer])} for layer in layers}


writes_unlocked = get_writes(model, tok, prompt_sets["unlocked"], range(6),
                             unit6, check=True)
writes_locked = get_writes(model, tok, prompt_sets["locked"], range(6), unit6)

writes_unlocked_9 = get_writes(
    model, tok, prompt_sets["unlocked"], range(9), unit9)
writes_locked_9 = get_writes(
    model, tok, prompt_sets["locked"], range(9), unit9)

for tag, upto, wu, wl, norm in [
        ("r6", 6, writes_unlocked, writes_locked, float(norms[6])),
        ("r9", 9, writes_unlocked_9, writes_locked_9, float(norms[9]))]:
    total = 0.0
    for layer in range(upto):
        diff_heads = (wu[layer]["heads"].mean(0) - wl[layer]["heads"].mean(0))
        diff_mlp = float(wu[layer]["mlp"].mean() - wl[layer]["mlp"].mean())
        total += float(diff_heads.sum()) + diff_mlp
        top = diff_heads.abs().topk(2)
        tops = ", ".join(f"H{int(i)} {float(diff_heads[i]):+.2f}"
                         for i in top.indices)
        print(f"L{layer}: attention {float(diff_heads.sum()):+6.2f}  ({tops})"
              f"mlp {diff_mlp:+6.2f}")

In [ ]:
def get_password_spans(problem, password):
    """
    Get the input token span of the password in the prompt for a problem.
    """
    full = prompts.make_prompt(problem, tok, password)
    ids = tok(full).input_ids
    hits = []
    for pwd_ids in [tok(password, add_special_tokens=False).input_ids,
                    tok(" " + password, add_special_tokens=False).input_ids]:
        n = len(pwd_ids)
        for i in range(len(ids) - n + 1):
            if ids[i:i + n] == pwd_ids:
                hits.append((i, i + n))
    hits = sorted(set(hits))
    assert len(hits) >= 2, "password should appear twice, span search failed"
    spans = [hits[0], hits[-1]]
    assert spans[0][1] <= spans[1][0], f"spans overlap: {spans}"
    return full, spans


def score_severed(sample, layers):
    """
    Score the model on a sample of problems with the layers severed.
    """
    generations = []
    for prob in sample:
        full, spans = get_password_spans(prob["problem"], config.PASSWORD)
        fwd = [(model.model.layers[l].self_attn.v_proj, get_span_zero_hook(spans))
               for l in layers]
        with add_hooks([], fwd):
            generations += generate(model, tok, [full])
    acc, stderr = evaluate.accuracy(sample, generations)
    return float(acc), float(stderr)


sub_32 = data.subset(problems, splits, "val")[:32]
floor_32, _, _ = helpers.score(model, tok, sub_32, "locked")
ceiling_32, _, _ = helpers.score(model, tok, sub_32, "unlocked")
print(f"floor {floor_32:.3f}  ceiling {ceiling_32:.3f}\n")

for name, layers in [("sever layers 0-5", range(0, 6)),
                     ("sever layers 6-29", range(6, config.N_LAYERS))]:
    acc, stderr = score_severed(sub_32, layers)
    print(f"{name:18s} {acc:.3f} +- {stderr:.3f}")

In [ ]:
# Final test run
test = data.subset(problems, splits, "test")[:500]
test_results = {}
locked_test, locked_se, _ = helpers.score(model, tok, test, "locked")
unlocked_test, unlocked_se, _ = helpers.score(model, tok, test, "unlocked")
test_results["baseline"] = {"locked": locked_test, "locked_se": locked_se,
                            "unlocked": unlocked_test, "unlocked_se": unlocked_se}
print(f"floor {locked_test:.3f}   ceiling {unlocked_test:.3f}\n")


def test_rho(acc):
    return (acc - locked_test) / (unlocked_test - locked_test)


def run(name, condition, pre=(), fwd=(), keep_example=False):
    acc, se, gens = helpers.score(model, tok, test, condition, pre, fwd)
    test_results[name] = {"acc": acc, "stderr": se, "rho": test_rho(acc)}
    if keep_example:
        test_results[name]["examples"] = [g[:400] for g in gens[:5]]
    helpers.save(f"experiment_{name}", test_results)
    return acc


d6 = d_none[0, 6].to(model.device, model.dtype)
d9 = d_none[0, 9].to(model.device, model.dtype)
clamp6 = [(model.model.layers[6], get_mean_matching_pre_hook(unit6, s6_unlocked))]
clamp9 = [(model.model.layers[9], get_mean_matching_pre_hook(unit9, s9_unlocked))]

run("clamp_d6_locked",   "locked",   clamp6)
run("clamp_d6_unlocked", "unlocked", clamp6)
run("clamp_d9_locked",   "locked",   clamp9)
run("clamp_d9_unlocked", "unlocked", clamp9)

run("add_d6_locked", "locked",
    [(model.model.layers[6], get_activation_addition_input_pre_hook(d6, 1.5))])
run("add_d9_locked", "locked",
    [(model.model.layers[9], get_activation_addition_input_pre_hook(d9, 1.0))])

for tag, vec in (("d6", d6), ("d9", d9)):
    abl_pre, abl_fwd = get_all_direction_ablation_hooks(model, vec)
    for cond in ("locked", "unlocked"):
        run(f"ablate_{tag}_{cond}", cond, abl_pre, abl_fwd)

run("relock_d6_a-1.5", "unlocked",
    [(model.model.layers[6], get_activation_addition_input_pre_hook(d6, -1.5))],
    keep_example=True)

# norm-matched random controls
for seed in range(3):
    g = torch.Generator().manual_seed(seed)
    rv = torch.randn(d6.shape, generator=g).to(d6)
    rv = rv / rv.norm() * d6.norm()
    run(f"random_{seed}", "locked",
        [(model.model.layers[6], get_activation_addition_input_pre_hook(rv, 1.5))])

helpers.save("experiment_random", test_results)